# M4 Assignment2 - Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL
>Part D: Final Model (Fine-Tune PatentSBERTa Once)

Suchanya Baiyam Business Data Science AAU


* Dataset:AI-Growth-Lab/patents_claims_1.5m_traim_test (The provided 50k samples)
* Model: AI-Growth-Lab/PatentSBERTa
* Working file: patents_50k_green.parquet with splits train_silver, pool_unlabeled, eval_silver
* Silver label: is_green_silver (derived from CPC Y02*)

Task:
- Merge the 100 HITL labels into your dataset to create is_green_gold (gold overrides silver for those 100 items).
- Fine-tune PatentSBERTa once for binary classification using train_silver + gold_100.
- Evaluate on eval_silver and also report results on gold_100.

In [ ]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


STEP01: Load libraries

In [1]:
import pandas as pd
import numpy as np
import evaluate
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

STEP02: Load Data

In [5]:
df = pd.read_parquet("patent_50k_green.parquet")
gold = pd.read_csv("gold_100_labeled.csv")

STEP03: Merge Gold

In [6]:
df = df.merge(
    gold[["doc_id", "is_green_human"]],
    on="doc_id",
    how="left"
)

df["is_green_final"] = df["is_green_human"].fillna(df["is_green_silver"])

STEP04: Prepare split

In [7]:
train_df = df[df["split"] == "train_silver"].copy()
eval_df  = df[df["split"] == "eval_silver"].copy()
gold_df = df[df["doc_id"].isin(gold["doc_id"])].copy()


In [8]:
#Assist by chatgpt to make labels in INT
train_df["is_green_final"] = train_df["is_green_final"].astype(int)
eval_df["is_green_final"]  = eval_df["is_green_final"].astype(int)
gold_df["is_green_final"]  = gold_df["is_green_final"].astype(int)

STEP05: Convert to HF Dataset

In [9]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

eval_dataset = Dataset.from_pandas(
    eval_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

gold_dataset = Dataset.from_pandas(
    gold_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

In [10]:
print(type(train_dataset[0]["label"]))


<class 'int'>


STEP06: Load Tokenizer + Model

In [11]:
model_name = "AI-Growth-Lab/PatentSBERTa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

MPNetForSequenceClassification LOAD REPORT from: AI-Growth-Lab/PatentSBERTa
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STEP07: Tokenizer (max_leghth = 256)

In [12]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)
gold_dataset = gold_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
#assist by chat gpt to remove text + set torch format
train_dataset = train_dataset.remove_columns(["text"])
eval_dataset  = eval_dataset.remove_columns(["text"])
gold_dataset  = gold_dataset.remove_columns(["text"])

train_dataset = train_dataset.remove_columns(["__index_level_0__"])
eval_dataset = eval_dataset.remove_columns(["__index_level_0__"])
gold_dataset = gold_dataset.remove_columns(["__index_level_0__"])

train_dataset.set_format("torch")
eval_dataset.set_format("torch")
gold_dataset.set_format("torch")

In [15]:
print(train_dataset.column_names)


['label', 'input_ids', 'attention_mask']


STEP08: Metric (F1)

In [26]:
import numpy as np
import evaluate

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = metric.compute(predictions=predictions, references=labels)
    return {"f1": f1["f1"]}


STEP09: Training Arguments (epoch1/ lr = 2e-5)

In [17]:
training_args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to = "none"
)

STEP10: Trainer

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics)

STEP11: Train

In [19]:
trainer.train()

Step,Training Loss
500,0.514561
1000,0.451308
1500,0.427728


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.45498494466145833, metrics={'train_runtime': 1469.5954, 'train_samples_per_second': 20.414, 'train_steps_per_second': 1.276, 'total_flos': 3946665830400000.0, 'train_loss': 0.45498494466145833, 'epoch': 1.0})

In [25]:
trainer.state

TrainerState(epoch=1.0, global_step=1875, max_steps=1875, logging_steps=500, eval_steps=500, save_steps=500, train_batch_size=16, num_train_epochs=1, num_input_tokens_seen=0, total_flos=3946665830400000.0, log_history=[{'loss': 0.5145607299804688, 'grad_norm': 4.372091770172119, 'learning_rate': 1.4677333333333334e-05, 'epoch': 0.26666666666666666, 'step': 500}, {'loss': 0.451307861328125, 'grad_norm': 5.125470161437988, 'learning_rate': 9.344e-06, 'epoch': 0.5333333333333333, 'step': 1000}, {'loss': 0.42772760009765626, 'grad_norm': 5.3571343421936035, 'learning_rate': 4.010666666666667e-06, 'epoch': 0.8, 'step': 1500}, {'train_runtime': 1469.5954, 'train_samples_per_second': 20.414, 'train_steps_per_second': 1.276, 'total_flos': 3946665830400000.0, 'train_loss': 0.45498494466145833, 'epoch': 1.0, 'step': 1875}], best_metric=None, best_global_step=None, best_model_checkpoint=None, is_local_process_zero=True, is_world_process_zero=True, is_hyper_param_search=False, trial_name=None, tri

STEP12: Evaluate (Evaluate on eval_silver and also report results on gold_100)
- Have new trainer to evaluate because the error when compute the metrix (typo)

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [28]:
print("Eval Silver:")
print(trainer.evaluate())

print("Gold 100:")
print(trainer.evaluate(gold_dataset))

Eval Silver:


{'eval_loss': 0.4348970353603363, 'eval_model_preparation_time': 0.0101, 'eval_f1': 0.8009847365829641, 'eval_runtime': 149.9259, 'eval_samples_per_second': 66.7, 'eval_steps_per_second': 4.169}
Gold 100:
{'eval_loss': 0.5585002899169922, 'eval_model_preparation_time': 0.0101, 'eval_f1': 0.5977011494252874, 'eval_runtime': 1.5258, 'eval_samples_per_second': 65.54, 'eval_steps_per_second': 4.588}


STEP13: Save model and tokenizer

In [29]:
trainer.save_model("PatentSBERTa_finetuned_green")
tokenizer.save_pretrained("PatentSBERTa_finetuned_green")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('PatentSBERTa_finetuned_green/tokenizer_config.json',
 'PatentSBERTa_finetuned_green/tokenizer.json')

# Upload model to Hungging Face
model + dataset
- https://huggingface.co/Ailee52/PatentSBERTa_finetuned_green
- https://huggingface.co/datasets/Ailee52/PatentSBERTa-green-dataset